# 🌾 Crop Recommendation Model Training
## Offline AI Assistant for Smart Farming
### Member 1 - AI/ML Module
---

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from src.preprocess import preprocess_crop_data
from src.crop_model import CropRecommendationModel

print('Libraries loaded ✅')

## Step 1: Preprocess Data

In [ ]:
X_train, X_test, y_train, y_test, scaler, le = preprocess_crop_data()
print(f'Classes: {list(le.classes_)}')

## Step 2: Train Models

In [ ]:
crop_model = CropRecommendationModel()
crop_model.label_encoder = le
crop_model.scaler = scaler
crop_model.train_random_forest(X_train, y_train)
crop_model.train_xgboost(X_train, y_train)

## Step 3: Evaluate Models

In [ ]:
results = crop_model.evaluate(X_test, y_test)

# Accuracy comparison bar chart
model_names = list(results.keys())
accuracies  = [results[m]['accuracy'] * 100 for m in model_names]

plt.figure(figsize=(7, 5))
bars = plt.bar(model_names, accuracies, color=['steelblue', 'coral'], edgecolor='white', width=0.5)
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 2,
             f'{acc:.2f}%', ha='center', va='top', color='white', fontweight='bold')
plt.ylim(90, 101)
plt.title('Crop Model Accuracy Comparison', fontsize=14)
plt.ylabel('Accuracy (%)')
plt.tight_layout()
plt.savefig('../data/processed/crop_accuracy_comparison.png', dpi=150)
plt.show()

## Step 4: Confusion Matrix

In [ ]:
# Confusion matrix for best model
best_name = max(results, key=lambda k: results[k]['accuracy'])
best_preds = results[best_name]['predictions']

cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix - {best_name}', fontsize=16)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/crop_confusion_matrix.png', dpi=150)
plt.show()

## Step 5: Feature Importance

In [ ]:
feature_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
importances  = crop_model.rf_model.feature_importances_

plt.figure(figsize=(9, 5))
sorted_idx = np.argsort(importances)[::-1]
plt.bar([feature_cols[i] for i in sorted_idx],
        [importances[i] for i in sorted_idx],
        color='steelblue', edgecolor='white')
plt.title('Feature Importance - Random Forest (Crop Model)', fontsize=14)
plt.ylabel('Importance')
plt.xlabel('Feature')
plt.tight_layout()
plt.savefig('../data/processed/crop_feature_importance.png', dpi=150)
plt.show()

## Step 6: Save Models

In [ ]:
crop_model.save_models()
print('✅ All crop models saved!')

## Step 7: Test Prediction

In [ ]:
test_input = {
    'N': 90, 'P': 42, 'K': 43,
    'temperature': 20.87, 'humidity': 82.0,
    'ph': 6.5, 'rainfall': 202.93
}

result = crop_model.predict(test_input)
print(f"Recommended Crop : {result['recommended_crop']}")
print(f"Confidence       : {result['confidence']}%")
print(f"Top 3 Crops      : {result['top_3_crops']}")